# Taller: Anatomía de un intérprete
## Análisis del código fuente de Picol

**Curso:** IS753 – Compiladores  
**Duración estimada:** 3 horas  
**Modalidad:** Individual o en parejas  
**Archivo requerido:** `picol.c`

---

## Propósito

En este taller se estudiará **Picol**, una implementación pequeña de un intérprete similar a Tcl escrita en C.

El objetivo es localizar en un programa real los componentes que normalmente aparecen en un compilador o intérprete:

1. Entrada del programa fuente.
2. Análisis léxico.
3. Análisis sintáctico.
4. Representación interna.
5. Gestión de variables y ámbitos.
6. Evaluación.
7. Ejecución de comandos.
8. Manejo de errores.
9. Gestión dinámica de memoria.

> Guarde `picol.c` en la misma carpeta de este cuaderno antes de ejecutar las celdas.

## Resultados de aprendizaje

Al finalizar el taller, el estudiante podrá:

- Diferenciar un compilador de un intérprete.
- Identificar el lexer y el parser dentro de un programa escrito en C.
- Explicar cómo Picol reconoce comandos, palabras, variables y bloques.
- Describir cómo se almacenan variables, procedimientos y comandos.
- Reconstruir el flujo de ejecución de un programa Tcl.
- Comparar la arquitectura de Picol con la de un compilador tradicional.

## 1. Preparación del entorno

In [1]:
from pathlib import Path
import re

ARCHIVO = Path("picol.c")

if not ARCHIVO.exists():
    raise FileNotFoundError(
        "No se encontró picol.c. Copie el archivo en la misma carpeta "
        "del cuaderno y vuelva a ejecutar esta celda."
    )

codigo = ARCHIVO.read_text(encoding="utf-8", errors="replace")
lineas = codigo.splitlines()

print(f"Archivo cargado: {ARCHIVO}")
print(f"Número de líneas: {len(lineas)}")
print(f"Número de caracteres: {len(codigo)}")

Archivo cargado: picol.c
Número de líneas: 809
Número de caracteres: 26727


### Pregunta 1

Antes de analizar el código, responda:

1. ¿Picol es principalmente un compilador, un intérprete o una máquina virtual?
2. ¿Qué diferencia existe entre compilar un programa e interpretarlo?
3. ¿Qué salida espera obtener Picol después de procesar un script?

**Respuesta:**

1. Picol es un interprete
2. Compilar un programa es traducir el código fuente y a partir de este generar un ejecutable, lo que quiere decir que ya no necesito el compilador, por otro lado el interprete: lee, traduce y ejecuta todas las instrucciones, por lo cual es dependiente del interprete
3. Al ser un interprete debera mostrar en pantalla lo que el codigo fuente pretendia mostrar 


## 2. Exploración general del archivo

In [2]:
for numero, linea in enumerate(lineas[:80], start=1):
    print(f"{numero:4}: {linea}")

   1: /* Tcl in ~ 500 lines of code.
   2:  *
   3:  * IMPORTANT: this is Picol version 2! For the original code, check
   4:  * the commit history of this repository.
   5:  *
   6:  * Copyright (c) 2007-2026, Salvatore Sanfilippo <antirez at gmail dot com>
   7:  * All rights reserved.
   8:  *
   9:  * Redistribution and use in source and binary forms, with or without
  10:  * modification, are permitted provided that the following conditions are met:
  11:  *
  12:  *   * Redistributions of source code must retain the above copyright notice,
  13:  *     this list of conditions and the following disclaimer.
  14:  *   * Redistributions in binary form must reproduce the above copyright
  15:  *     notice, this list of conditions and the following disclaimer in the
  16:  *     documentation and/or other materials provided with the distribution.
  17:  *
  18:  * THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
  19:  * AND ANY EXPRESS OR IMPLIED WARRANTIE

In [3]:
patron_funcion = re.compile(
    r"^\s*(?:static\s+)?(?:int|void|char\s*\*|const\s+char\s*\*|"
    r"struct\s+\w+\s*\*|\w+\s*\*)\s+"
    r"([A-Za-z_]\w*)\s*\([^;]*\)\s*\{",
    re.MULTILINE
)

funciones = patron_funcion.findall(codigo)

print(f"Funciones encontradas aproximadamente: {len(funciones)}")
for nombre in funciones:
    print("-", nombre)

Funciones encontradas aproximadamente: 28
- picolInitParser
- picolParseSep
- picolParseEol
- picolParseCommand
- picolParseVar
- picolParseBrace
- picolParseString
- picolParseComment
- picolGetToken
- picolSetResult
- picolSetVar
- picolRegisterCommand
- picolEval
- picolExprExpansion
- picolArityErr
- picolCommandExpr
- picolCommandSet
- picolCommandPuts
- picolCommandIf
- picolCommandWhile
- picolCommandRetCodes
- picolDropCallFrame
- picolFreeInterp
- picolCommandCallProc
- picolCommandProc
- picolCommandReturn
- picolRegisterCoreCommands
- main


### Actividad 2

Observe la lista de funciones y clasifíquelas inicialmente.

| Categoría | Funciones candidatas |
|---|---|
| Lexer o tokenización |picolGetToken |
| Parser |  picolInitParser, picolParseSep, picolParseEol, picolParseCommand, picolParseVar, picolParseBrace, picolParseString, picolParseComment|
| Evaluación | picolEval|
| Variables | picolSetVar|
| Procedimientos | picolCommandCallProc, picolCommandProc|
| Manejo de errores | picolCommandRetCodes, picolArityErr|
| Memoria | picolRegisterCommand|

> Esta primera clasificación es provisional. Será revisada al final del taller.

## 3. Herramientas para inspeccionar el código

In [4]:
def buscar(texto, contexto=3, ignorar_mayusculas=True):
    # Busca un texto o una expresión regular y muestra las líneas cercanas.
    flags = re.IGNORECASE if ignorar_mayusculas else 0
    patron = re.compile(texto, flags)

    encontrados = 0
    for i, linea in enumerate(lineas):
        if patron.search(linea):
            encontrados += 1
            inicio = max(0, i - contexto)
            fin = min(len(lineas), i + contexto + 1)

            print("=" * 78)
            for j in range(inicio, fin):
                marca = ">>" if j == i else "  "
                print(f"{marca} {j + 1:4}: {lineas[j]}")

    if encontrados == 0:
        print("No se encontraron coincidencias.")
    else:
        print(f"\nCoincidencias: {encontrados}")


def mostrar_rango(inicio, fin):
    # Muestra un rango de líneas. Los números se interpretan desde 1.
    inicio = max(1, inicio)
    fin = min(len(lineas), fin)

    for numero in range(inicio, fin + 1):
        print(f"{numero:4}: {lineas[numero - 1]}")

In [5]:
buscar(r"picolEval", contexto=4)

    378:     }
    379: }
    380: 
    381: /* EVAL! */
>>  382: int picolEval(struct picolInterp *i, char *t) {
    383:     struct picolParser p;
    384:     int argc = 0, j;
    385:     char **argv = NULL;
    386:     char errbuf[1024];
    414:             }
    415:             free(t);
    416:             t = xstrdup(v->val);
    417:         } else if (p.type == PT_CMD) {
>>  418:             retcode = picolEval(i,t);
    419:             free(t);
    420:             if (retcode != PICOL_OK) goto err;
    421:             t = xstrdup(i->result);
    422:         } else if (p.type == PT_ESC) {
    553:     i->level--;
    554:     return a;
    555: }
    556: 
>>  557: /* Trick: wrap 's' as "expr <s>" and evaluate it, so that picolEval handles
    558:  * $var and [cmd] substitution before expr parses pure math expression.
    559:  * This is used in [if] and [while] condition evaluation. */
    560: int picolExprExpansion(struct picolInterp *i, char *s) {
    561:     int

## 4. Identificación del lexer

Un **lexer** o analizador léxico recibe caracteres y reconoce unidades significativas.

En Picol estas unidades pueden incluir:

- Palabras.
- Separadores.
- Variables.
- Cadenas entre comillas.
- Bloques entre llaves.
- Sustitución de comandos.
- Finales de comando.

In [6]:
terminos_lexer = [
    r"parse",
    r"token",
    r"separator",
    r"brace",
    r"quote",
    r"variable",
    r"command",
    r"eol",
]

for termino in terminos_lexer:
    print("\n" + "#" * 78)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=1)


##############################################################################
BÚSQUEDA: parse
     73: 
>>   74: struct picolParser {
     75:     char *text;         // The program to parse
     74: struct picolParser {
>>   75:     char *text;         // The program to parse
     76:     char *p;            // Current parsing position in 'text'
    113: 
>>  114: void picolInitParser(struct picolParser *p, char *text) {
    115:     p->text = p->p = text;
    120: 
>>  121: int picolParseSep(struct picolParser *p) {
    122:     p->start = p->p;
    130: 
>>  131: int picolParseEol(struct picolParser *p) {
    132:     p->start = p->p;
    142: 
>>  143: int picolParseCommand(struct picolParser *p) {
    144:     int level = 1;
    172: 
>>  173: int picolParseVar(struct picolParser *p) {
    174:     p->start = ++p->p; p->len--; /* skip the $ */
    192: 
>>  193: int picolParseBrace(struct picolParser *p) {
    194:     int level = 1;
    215: 
>>  216: int picolParseString(struc

### Actividad 4

Complete la tabla con los nombres reales encontrados en `picol.c`.

| Parte léxica | Función o constante | Explicación |
|---|---|---|
| Inicio del parser | picolInitParser| recibe el texto que el parser evaluara|
| Separadores | picolParseSep| separa argumentos|
| Palabras simples | picolParseString| evalua tokens que pueden ser de tipo PT_SEP, PT_EOL o PT_str toma acciones si es '{' o '"'|
| Variables | picolParseVar| se salta el primer y caracter de una string|
| Comillas | picolparseQuote| Evalua si hay algo entre comillas|
| Llaves | picolparseQuote| Evalua si hay algo entre llaves|
| Sustitución de comandos | picolRegisterCommand| registra un comando, salta al siguiente si no existe, si no hay argumentos pasados por consola pone el resultado como un error de buffer|
| Fin de línea o comando | EOL| indica cuando hay un salto de linea o termina un comando|

Responda:

1. ¿Picol crea objetos `Token` independientes?
creo que todo hace parte de una estructura general, 
2. ¿El lexer genera primero una lista completa de tokens?
se supone deberia hacerlos primero
3. ¿Lexer y parser están completamente separados?
el parser depende del lexer, el uno sin el otro no son nada, uno hace tokenizacion y el otro analiza si estos estan organizados
4. ¿Qué información guarda el estado del parser?
recuerda que ya ha procesado

## 5. Identificación del parser

In [7]:
buscar(r"picolGetToken", contexto=12)

    257:         p->p++; p->len--;
    258:     }
    259:     return PICOL_OK; /* unreached */
    260: }
    261: 
    262: int picolParseComment(struct picolParser *p) {
    263:     while(p->len && *p->p != '\n') {
    264:         p->p++; p->len--;
    265:     }
    266:     return PICOL_OK;
    267: }
    268: 
>>  269: int picolGetToken(struct picolParser *p) {
    270:     while(1) {
    271:         if (!p->len) {
    272:             if (p->type != PT_EOL && p->type != PT_EOF)
    273:                 p->type = PT_EOL;
    274:             else
    275:                 p->type = PT_EOF;
    276:             return PICOL_OK;
    277:         }
    278:         switch(*p->p) {
    279:         case ' ': case '\t':
    280:             if (p->insidequote) return picolParseString(p);
    281:             return picolParseSep(p);
    387:     int retcode = PICOL_OK;
    388:     picolSetResult(i,"");
    389:     if (++i->level > PICOL_MAX_RECURSION_LEVEL) {
    390:         i->l

In [8]:
buscar(r"picolParse", contexto=8)

     66:     PT_STR, // String without escapes, no post processing needed.
     67:     PT_CMD, // Command, that is [.... something ...]
     68:     PT_VAR, // Variable like $var
     69:     PT_SEP, // Arguments separator
     70:     PT_EOL, // End of command
     71:     PT_EOF  // End of file (stops the parsing loop)
     72: };
     73: 
>>   74: struct picolParser {
     75:     char *text;         // The program to parse
     76:     char *p;            // Current parsing position in 'text'
     77:     int len;            // Remaining length
     78:     char *start;        // Token start
     79:     char *end;          // Token end
     80:     int type;           // Token type, PT_...
     81:     int insidequote;    // True if inside " "
     82: };
    106: 
    107: struct picolInterp {
    108:     int level; /* Level of nesting */
    109:     struct picolCallFrame *callframe;
    110:     struct picolCmd *commands;
    111:     char *result;
    112: };
    113: 
>>  

### Actividad 5

Explique el flujo del parser:

```text
Texto fuente
    ↓
despues de que se recibe el texto fuente entra a picolInitParser()
    ↓
pasa a picolGetToken() generando los tokens
    ↓
Token o fragmento reconocido
    ↓
los tokens pasaran a picolEval()
```

¿Picol usa una gramática BNF explícita como Yacc, Bison o SLY?  
no, esta todo escrito "a mano".
Explique la diferencia entre un parser generado y el parser manual de Picol.
El parser generado requiere que uno genere la gramatica, en el manual todo se puede hacer con estructuras de control

## 6. ¿Existe un árbol de sintaxis abstracta?

In [9]:
for termino in [r"AST", r"Node", r"Statement", r"Expression", r"Program"]:
    print("\n" + "-" * 60)
    print(f"Búsqueda de: {termino}")
    buscar(termino, contexto=1)


------------------------------------------------------------
Búsqueda de: AST
No se encontraron coincidencias.

------------------------------------------------------------
Búsqueda de: Node
No se encontraron coincidencias.

------------------------------------------------------------
Búsqueda de: Statement
    644:         if (j >= argc) return PICOL_OK; // No more branches.
>>  645:         /* Else statement? Evaluate the else branch (condition was false)
    646:          * if we are here. */

Coincidencias: 1

------------------------------------------------------------
Búsqueda de: Expression
    489: 
>>  490: /* This is a "Pratt style parser" for expressions: precedence is encoded in a
    491:  * single recursive function. Basically the C call stack replaces the explicit
    505:         i->level--;
>>  506:         *err = 1; // Will be reported as error in expression, instead of
    507:         return 0; // recursion limit. Requires a pathological expression anyway.
    506:

### Actividad 6

Responda:

1. ¿Existe una estructura que represente un AST completo?
no
2. ¿Picol analiza primero todo el programa y lo ejecuta después?
no, eval se hace al tiempo que tokeniza
3. ¿Qué ventaja obtiene al no construir un AST?
es mas simple hay menos codigo
4. ¿Qué limitaciones produce esta decisión?
es menos escalable, como vimos en la clase esta peor optimizado que el interprete de tlc
5. ¿La lista de palabras de un comando puede considerarse una representación intermedia mínima?


## 7. El ciclo de evaluación

In [10]:
buscar(r"int\s+picolEval|picolEval\s*\(", contexto=25)

    357: 
    358: void picolRegisterCommand(struct picolInterp *i, char *name, picolCmdFunc f) {
    359:     struct picolCmd *c = picolGetCommand(i,name);
    360:     int existing = c != NULL;
    361: 
    362:     if (!existing) {
    363:         c = xmalloc(sizeof(*c));
    364:         c->name = NULL;
    365:         c->arglist = NULL;
    366:         c->body = NULL;
    367:     } else {
    368:         free(c->arglist);
    369:         free(c->body);
    370:         c->arglist = NULL;
    371:         c->body = NULL;
    372:     }
    373:     if (!c->name) c->name = xstrdup(name);
    374:     c->func = f;
    375:     if (!existing) {
    376:         c->next = i->commands;
    377:         i->commands = c;
    378:     }
    379: }
    380: 
    381: /* EVAL! */
>>  382: int picolEval(struct picolInterp *i, char *t) {
    383:     struct picolParser p;
    384:     int argc = 0, j;
    385:     char **argv = NULL;
    386:     char errbuf[1024];
    387:     int retc

### Actividad 7

Describa paso a paso la función de evaluación.

Después explique qué ocurre al ejecutar:

```tcl
set x 10
puts $x
```

Incluya:

- Lectura del comando `set`.
- Construcción de argumentos.
- Almacenamiento de `x`.
- Lectura de `puts`.
- Sustitución de `$x`.
- Búsqueda y ejecución de `puts`.

RESPUESTA:
Se llama a picolEval, la función crea la estructura p de tipo picolParser, define el argument counter y vector como 0 y nulo respectivamente.
Limpia el resultado y aumenta el nivel de recursion. Ahora es cuando inicializa el parser con picolInitParser, el cual apuntara al primer caracter del texto, empieza el ciclo de obtención de tokens, con cada token decide a que función llamar y deja dos punteros p.start, p.end
que es donde inicia y termina cada token, luego procesa el token.
Cuando se le entrega set x 10 al interprete lo que sucede es: el token anterior es EOL, por lo tanto set es el primer elemento del vector de argumentos, el argument count aumenta en 1, lee el espacio, luego x, se agrega a argv y aumenta argc, 10 es el tercer argumento, aqui termina la linea. Se busca el comando set, lo encuentra y lo ejecuta, x se almacena con el valor de 10, se liberan argv.
La siguiente orden es puts la cual sera el primer argumento, se busca x, se encuentra su valor 10 y se reemplaza el token, eso se le entregara al buscador de comandos para finalmente imprimir 10 

## 8. Tabla de comandos

In [11]:
buscar(r"picolRegister", contexto=12)
buscar(r"struct\s+picolCmd|picolCmd", contexto=8)

    346:     }
    347: }
    348: 
    349: struct picolCmd *picolGetCommand(struct picolInterp *i, char *name) {
    350:     struct picolCmd *c = i->commands;
    351:     while(c) {
    352:         if (strcmp(c->name,name) == 0) return c;
    353:         c = c->next;
    354:     }
    355:     return NULL;
    356: }
    357: 
>>  358: void picolRegisterCommand(struct picolInterp *i, char *name, picolCmdFunc f) {
    359:     struct picolCmd *c = picolGetCommand(i,name);
    360:     int existing = c != NULL;
    361: 
    362:     if (!existing) {
    363:         c = xmalloc(sizeof(*c));
    364:         c->name = NULL;
    365:         c->arglist = NULL;
    366:         c->body = NULL;
    367:     } else {
    368:         free(c->arglist);
    369:         free(c->body);
    370:         c->arglist = NULL;
    740: arityerr:
    741:     snprintf(errbuf,sizeof(errbuf),"Proc '%s' called with wrong arg num",argv[0]);
    742: err:
    743:     picolSetResult(i,errbuf);
    7

### Actividad 8

| Elemento | Nombre en Picol | Función |
|---|---|---|
| Estructura de comando |picolCmd | Almacenar toda la informacion referente a un comando, nombre, funcion, lista de argumentos|
| Registro de comando | picolRegisterCommand| Registra un comando en el interprete|
| Búsqueda de comando | picolGetCommand| Busca un argumento por su nombre|
| Función C asociada | func| Emula el comportamiento del comando|
| Datos privados | arglist| Contiene los nombres de los parámetros del procedimiento.|

Responda cómo se asocia el nombre de un comando con una función de C y qué sucede cuando el comando no existe.
Se guarda mediante picolCmd, con  picolRegisterCommand, si no existe retorna No such command 


## 9. Variables y ámbitos

In [12]:
for termino in [r"picolVar", r"picolGetVar", r"picolSetVar", r"callframe", r"CallFrame"]:
    print("\n" + "#" * 70)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=6)


######################################################################
BÚSQUEDA: picolVar
     78:     char *start;        // Token start
     79:     char *end;          // Token end
     80:     int type;           // Token type, PT_...
     81:     int insidequote;    // True if inside " "
     82: };
     83: 
>>   84: struct picolVar {
     85:     char *name, *val;
     86:     struct picolVar *next;
     87: };
     88: 
     89: struct picolInterp;     // Forward declarations
     90: struct picolCmd;
     80:     int type;           // Token type, PT_...
     81:     int insidequote;    // True if inside " "
     82: };
     83: 
     84: struct picolVar {
     85:     char *name, *val;
>>   86:     struct picolVar *next;
     87: };
     88: 
     89: struct picolInterp;     // Forward declarations
     90: struct picolCmd;
     91: typedef int (*picolCmdFunc)(struct picolInterp *i, int argc, char **argv, struct picolCmd *cmd);
     92: 
     97:     // Aux data for user def

### Actividad 9

Complete el diagrama:

```text
Intérprete
    ↓
Marco de llamada actual
    ↓
Lista de variables
    ↓
Nombre siguiente Valor
```

Responda:

1. ¿Cómo se representa una variable?
2. ¿Se usa un arreglo, tabla hash o lista enlazada?
3. ¿Cómo se implementan los ámbitos locales?
4. ¿Qué ocurre cuando se llama un procedimiento?

RESPUESTAS:
1. con picolVar tiene nombre valor siguiente
2. El primer elemento esta en: picolCallFrame y se recorre uno a uno y si se encuentra se retorna v
3. con marcos que contienen variables y el marco anterior
4. se crea el procedimiento, se busca, se crea un nuevo marco, se asocian parametros y argumentos, se evalua el body, se retorna el resultado, se libera el marco

## 10. Procedimientos definidos por el usuario

In [20]:
for termino in [r"picolProc", r"proc", r"picolCallProc"]:
    print("\n" + "#" * 70)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=10)


######################################################################
BÚSQUEDA: picolProc
No se encontraron coincidencias.

######################################################################
BÚSQUEDA: proc
     13:  *     this list of conditions and the following disclaimer.
     14:  *   * Redistributions in binary form must reproduce the above copyright
     15:  *     notice, this list of conditions and the following disclaimer in the
     16:  *     documentation and/or other materials provided with the distribution.
     17:  *
     18:  * THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
     19:  * AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
     20:  * IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE
     21:  * ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT OWNER OR CONTRIBUTORS BE
     22:  * LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR
>>   23:  * CONSEQUENTIAL DAM

### Actividad 10

Analice:

```tcl
proc cuadrado {x} {
    expr {$x * $x}
}
```

| Elemento del procedimiento | Representación en Picol |
|---|---|
| Nombre | cmd->name = "cuadrado";|
| Lista de parámetros | cmd->arglist = "x";|
| Cuerpo | cmd->body = "expr {$x * $x}";|
| Datos privados del comando |  name, func, next, arglist, body|
| Marco de llamada | picolCallFrame|

Explique cuándo se interpreta el cuerpo y cómo se asignan argumentos a parámetros.
Al definir el procedimiento, La función recorre la lista de parámetros. Para cada parámetro: obtiene su nombre, toma el argumento correspondiente de argv, crea la variable local con picolSetVar().


## 11. Comandos incorporados

In [14]:
for termino in [
    r"picolCommandSet",
    r"picolCommandPuts",
    r"picolCommandIf",
    r"picolCommandWhile",
    r"picolCommandProc",
    r"picolCommandReturn",
]:
    print("\n" + "=" * 70)
    print(termino)
    buscar(termino, contexto=8)


picolCommandSet
    595:     if (*p != '\0') err = 1;
    596:     free(expr);
    597:     if (err) { picolSetResult(i,"Error in expression"); return PICOL_ERR; }
    598:     snprintf(buf,sizeof(buf),"%.12g",v);
    599:     picolSetResult(i,buf); return PICOL_OK;
    600: }
    601: 
    602: /* set var ?value? */
>>  603: int picolCommandSet(struct picolInterp *i, int argc, char **argv, struct picolCmd *cmd) {
    604:     if (argc == 3) {
    605:         picolSetVar(i,argv[1],argv[2]);
    606:         picolSetResult(i,argv[2]);
    607:     } else if (argc == 2) {
    608:         struct picolVar *v = picolGetVar(i,argv[1]);
    609:         if (v == NULL) {
    610:             char buf[1024];
    611:             snprintf(buf,sizeof(buf),
    759: int picolCommandReturn(struct picolInterp *i, int argc, char **argv, struct picolCmd *cmd) {
    760:     if (argc != 1 && argc != 2) return picolArityErr(i,argv[0]);
    761:     picolSetResult(i, (argc == 2) ? argv[1] : "");
    7

### Actividad 11

| Comando | Función C | Validación de argumentos | Operación |
|---|---|---|---|
| `set` | picolCommandSet| se valida la cantidad de argumentos, 2 o 3| buscar la variable, devuelve su valor si existe|
| `if` | picolCommandIf| se valida la cantidad de argumentos, 3, se  evalua con picolExprExpansion| ejecuta el cuerpo con picolEval|
| `while` | picolCommandWhile| se valida la cantidad de argumentos, 3| evalúa la condición, si es falsa, termina, si es verdadera, evalúa el cuerpo, vuelve al paso 1|

¿`if` y `while` forman parte del parser o se implementan como comandos normales? Explique.
son comandos normales implementados, registran en picolRegisterCoreCommands Lo que sucede es que se construye argv busca el nombre del comando, valida y ejecuta la operación

## 12. Expresiones

In [15]:
buscar(r"expr", contexto=10)

      9:  * Redistribution and use in source and binary forms, with or without
     10:  * modification, are permitted provided that the following conditions are met:
     11:  *
     12:  *   * Redistributions of source code must retain the above copyright notice,
     13:  *     this list of conditions and the following disclaimer.
     14:  *   * Redistributions in binary form must reproduce the above copyright
     15:  *     notice, this list of conditions and the following disclaimer in the
     16:  *     documentation and/or other materials provided with the distribution.
     17:  *
     18:  * THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
>>   19:  * AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
     20:  * IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE
     21:  * ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT OWNER OR CONTRIBUTORS BE
     22:  * LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL

### Actividad 12

1. ¿Existe un parser independiente para expresiones?
2. ¿Qué operadores soporta?
3. ¿Existe precedencia?
4. ¿Cómo se procesan los operandos?
5. ¿La expresión se compila o se evalúa directamente?
RESPUESTA:
1. Si pero es manual: picolExpr
2. suma, resta, multiplicación, division
3. Si se implementa mediante prec que es un entero
4. Los operandos numéricos se leen con: strtod, lo que convierte el texto en numeros.
si es parentesis, se analiza recursivamente una expresion completa y se espera un cierre.
Si son unarios se evalua si es - para calcular la negacion
Si es un operando invalido se marca como expresion invalida   
5. se evalua directamente


## 13. Manejo de errores y códigos de retorno

In [16]:
for termino in [
    r"PICOL_OK",
    r"PICOL_ERR",
    r"PICOL_RETURN",
    r"PICOL_BREAK",
    r"PICOL_CONTINUE",
    r"wrong # args",
    r"unknown command",
]:
    print("\n" + "#" * 70)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=4)


######################################################################
BÚSQUEDA: PICOL_OK
     59:  * ========================================================================== */
     60: 
     61: #define PICOL_MAX_RECURSION_LEVEL 128
     62: 
>>   63: enum {PICOL_OK, PICOL_ERR, PICOL_RETURN, PICOL_BREAK, PICOL_CONTINUE};
     64: enum {
     65:     PT_ESC, // String that may contain escapes (that should be processed)
     66:     PT_STR, // String without escapes, no post processing needed.
     67:     PT_CMD, // Command, that is [.... something ...]
    124:         p->p++; p->len--;
    125:     }
    126:     p->end = p->p-1;
    127:     p->type = PT_SEP;
>>  128:     return PICOL_OK;
    129: }
    130: 
    131: int picolParseEol(struct picolParser *p) {
    132:     p->start = p->p;
    136:         p->p++; p->len--;
    137:     }
    138:     p->end = p->p-1;
    139:     p->type = PT_EOL;
>>  140:     return PICOL_OK;
    141: }
    142: 
    143: int picolParseCommand

### Actividad 13

| Código | Significado | Ejemplo |
|---|---|---|
| `PICOL_OK` | Indica que el comando terminó correctamente| set x 10|
| `PICOL_ERR` | Indica que ocurrió un error durante la evaluación| set (cantidad de argumentos erronea)|
| `PICOL_RETURN` | Indica que el comando return solicitó salir del procedimiento actual| return terminado|
| `PICOL_BREAK` | Indica que se solicitó salir del ciclo actual| while $x < 10 break|
| `PICOL_CONTINUE` | Indica que se debe terminar la iteración actual y comenzar la siguiente del ciclo| while $x > 10 continue|

Explique cómo se almacena y propaga un error.
Se almacena con PICOL_ERR. El evaluador no continúa ejecutando el resto del comando o del body actual

## 14. Gestión de memoria

In [17]:
for termino in [r"malloc", r"calloc", r"realloc", r"free", r"strdup"]:
    print("\n" + "#" * 70)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=2)


######################################################################
BÚSQUEDA: malloc
     46: }
     47: 
>>   48: #define xmalloc(size) xrealloc(NULL,size)
     49: 
     50: char *xstrdup(const char *s) {
     50: char *xstrdup(const char *s) {
     51:     size_t l = strlen(s);
>>   52:     char *dup = xmalloc(l+1);
     53:     memcpy(dup,s,l+1);
     54:     return dup;
    305: 
    306: struct picolInterp *picolInitInterp(void) {
>>  307:     struct picolInterp *i = xmalloc(sizeof(*i));
    308:     i->level = 0;
    309:     i->callframe = xmalloc(sizeof(struct picolCallFrame));
    307:     struct picolInterp *i = xmalloc(sizeof(*i));
    308:     i->level = 0;
>>  309:     i->callframe = xmalloc(sizeof(struct picolCallFrame));
    310:     i->result = xstrdup("");
    311:     i->callframe->vars = NULL;
    337:         v->val = xstrdup(val);
    338:     } else {
>>  339:         v = xmalloc(sizeof(*v));
    340:         v->name = xstrdup(name);
    341:         v->val =

### Actividad 14

| Estructura o dato | Reserva | Liberación | Riesgo |
|---|---|---|---|
| picolInterp| picolInitInterp| picolFreeInterp| No se debe modificar ni liberar externamente sin actualizar el puntero|
| picolCmd| picolRegisterCommand| picolFreeInterp| La búsqueda se realiza en una lista enlazada, por lo que puede volverse lenta |
| callFrame| xmalloc| picolDropCallFrame| El marco global debe ser el último marco eliminado.|
| picolVar| picolSetVar| picolDropCallFrame| Las variables se buscan linealmente|

Identifique los datos que permanecen durante toda la vida del intérprete y los que solo existen durante una llamada.
En picolEval se declara p la cual se guarda en la pila de c, contiene text, p,len, start, end, type, insidequote.Por lo cual solo estan vivas durante esa función.
Argv mientras se analiza el comando.
Se copian los tokens de manera temporal.


## 15. Función principal e inicialización

In [18]:
buscar(r"main\s*\(", contexto=25)

for termino in [r"picolInitInterp", r"picolRegister", r"argc", r"argv"]:
    print("\n" + "#" * 70)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=8)

    757: }
    758: 
    759: int picolCommandReturn(struct picolInterp *i, int argc, char **argv, struct picolCmd *cmd) {
    760:     if (argc != 1 && argc != 2) return picolArityErr(i,argv[0]);
    761:     picolSetResult(i, (argc == 2) ? argv[1] : "");
    762:     return PICOL_RETURN;
    763: }
    764: 
    765: void picolRegisterCoreCommands(struct picolInterp *i) {
    766:     picolRegisterCommand(i,"expr",picolCommandExpr);
    767:     picolRegisterCommand(i,"set",picolCommandSet);
    768:     picolRegisterCommand(i,"puts",picolCommandPuts);
    769:     picolRegisterCommand(i,"if",picolCommandIf);
    770:     picolRegisterCommand(i,"while",picolCommandWhile);
    771:     picolRegisterCommand(i,"break",picolCommandRetCodes);
    772:     picolRegisterCommand(i,"continue",picolCommandRetCodes);
    773:     picolRegisterCommand(i,"proc",picolCommandProc);
    774:     picolRegisterCommand(i,"return",picolCommandReturn);
    775: }
    776: 
    777: /* ===================

### Actividad 15

Complete:

```text
main
  ↓
Creación del intérprete
  ↓
inicializacion del interprete
  ↓
Registro de comandos
  ↓
Lectura del script
  ↓
evaluacion
  ↓
Presentación del resultado
```

Explique cómo se usan `argc` y `argv`.
Se usan para ejecutar los comandos, argc para comprobar que halla la cantidad correcta de argumentos, argv la lista de estos,

## 16. Correspondencia con las fases de un compilador

| Fase tradicional | ¿Existe? | Función, estructura o mecanismo | Observaciones |
|---|---:|---|---|
| Lectura del programa | si| fgets(clibuf, 1024, stdin)| Picol no lo analiza completamente antes de ejecutar|
| Análisis léxico | si| picolGetToken()|  Los tokens se producen uno por uno y se consumen inmediatamente|
| Análisis sintáctico | si| picolEval| El analisis sintáctico se realiza mientras se construye y ejecuta el comando|
| AST | no| argc y v| |
| Tabla de símbolos | no| picolCallFrame y Var| |
| Análisis semántico | no| | se hace dentro de las funciones picolCommand|
| Representación intermedia | si| argv y c| Solo existe mientras se construye y ejecuta el comando actual|
| Optimización | no| | |
| Generación de código | no| | |
| Máquina virtual | no| | |
| Ejecución directa | si| func| Es el modelo principal de picol y se realiza despues de construir su lista de palabras|
| Manejo de errores | si| PICOL_ERR, picolSetResult| no hay excepciones u objetos de error|

### Pregunta central

¿Por qué Picol puede ejecutar programas sin tener todas las fases de un compilador tradicional?
Picol puede ejecutar programas sin las fases completas de un compilador tradicional porque no intenta transformar todo el programa en código ejecutable previamente. En lugar de eso, interpreta cada comando conforme lo encuentra.
Un compilador tradicional seguiria los siguientes pasos:
-Leer programa fuente
-generar los tokens, que seria tarea de un lexer
-Crear AST
-Realizar analisis semantico
-Generar una representacion intermedia
-Llevar a cabo una optimizacion
-Generar el codigo de maquina
-ejecutar
Picol es mucho mas compacto y realiza varias tareas en una sola etapa ademas de que se salta otras.
El ejemplo perfecto: Al no llevar a cabo la optimizacion no requiere un AST.
Por otro lado no requiere de una maquina virtual ya que usa funciones de c para emular las funciones de tcl, en otra palabras no produce bytecode.
Al ser compacto recibe beneficios:
-Tiene una estructura menor
-Usa menos memoria
-Ejecuta de manera inmediata
Pero a su vez tiene desventajas:
-No detecta errores antes de iniciar
-No optimiza globalmente
-No conserva las instrucciones generadas
-el cuerpo de un procedimiento debe reinterpretarse cada vez que se llama
-Escalar el interprete seria complejo
Picol puede ejecutar programas sin todas las fases de un compilador tradicional porque es un intérprete directo. No transforma primero el programa completo; procesa una orden, prepara sus argumentos, ejecuta la función asociada y continúa con la siguiente.
No elimina completamente el análisis: simplemente lo combina con la evaluación y lo realiza bajo demanda.
**Extensión mínima:** 200 palabras.

## 17. Diagrama de arquitectura

Complete con nombres reales:

```text
                         Script Tcl
                             │
                             ▼
                   Estado del parser
                   __________________
                             │
                             ▼
                 Reconocimiento de palabras
                            
                             │
                             ▼
               Sustitución de variables/comandos
               ________________________________
                             │
                             ▼
                   Lista de argumentos
                   __________________
                             │
                             ▼
                   Búsqueda del comando
                   __________________
                             │
                             ▼
                  Función C o procedimiento
                  _________________________
                             │
                             ▼
                     Resultado o error
```

## 18. Prueba experimental

Compile desde una terminal:

```bash
gcc -Wall -Wextra -O0 -g picol.c -o picol
```

Registre:

- Comando de compilación.
- Advertencias.
warning: unused parameter
- Comando de ejecución.
- Salida.
No dio salida
- Errores encontrados.
No encontro erroes

In [19]:
print("gcc -Wall -Wextra -O0 -g picol.c -o picol")

gcc -Wall -Wextra -O0 -g picol.c -o picol


## 19. Extensión opcional: comando `square`

Implemente:

```tcl
square 10
```

Resultado:

```text
100
```

Debe entregar:

1. Función C.
2. Registro del comando.
3. Prueba.
4. Explicación.
5. Manejo de argumentos inválidos.

## 20. Preguntas de reflexión

1. ¿Qué parte corresponde al lexer?
2. ¿Qué parte corresponde al parser?
3. ¿Por qué están tan unidos?
4. ¿Qué estructura actúa como tabla de símbolos?
5. ¿Cómo se representan los procedimientos?
6. ¿Por qué `if` y `while` pueden ser comandos?
7. ¿Qué se gana al interpretar directamente?
8. ¿Qué se pierde al no construir un AST?
9. ¿Qué cambios permitirían convertir Picol en compilador?
10. ¿Cuál fue la parte más difícil de identificar?
RESPUESTAS
1. esta representado principalmente por picolParser, picolGetToken y las funciones con picolParse
2. Corresponde a picolEval
3. Porque es pequeño y directo, no necesita conservar todos los tokens
4. i-commands y picolCmd
5. son comandos especiales dentro de picolCmd 
6. porque estan basadas en comandos, son nombres registrados asociados con funciones C
7. evita pasos como generar bytecode y reduce el uso de la memoria
8. Optimizacion
9. Lo principal seria dejar de hacer evaluación inmediata basada en texto y llamadas C a una representación persistente del programa que pueda analizar, transformar y ejecutar posteriormente
10. Las expresiones

## 21. Entregables y rúbrica

### Entregables

1. Cuaderno ejecutado y respondido.
2. Archivo `picol.c`.
3. Diagrama de arquitectura.
4. Tabla de fases.
5. Flujo de `set x 10; puts $x`.
6. Extensión `square`, cuando sea solicitada.

### Rúbrica

| Criterio | Porcentaje |
|---|---:|
| Identificación del lexer | 20 % |
| Identificación del parser | 20 % |
| Ciclo de evaluación | 20 % |
| Variables, ámbitos y procedimientos | 15 % |
| Comparación con un compilador | 15 % |
| Claridad y presentación | 10 % |
| **Total** | **100 %** |

## Conclusión del estudiante

Redacte una conclusión general sobre la arquitectura de Picol y su utilidad para aprender compiladores e intérpretes.

**Extensión mínima:** 250 palabras.